In [ ]:
import duckdb
import os
from dotenv import load_dotenv

load_dotenv("../.env")

conn = duckdb.connect()
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

s3_endpoint = "localhost:9000"
access_key = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin")

conn.execute(f"""
CREATE SECRET (
    TYPE S3,
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    ENDPOINT '{s3_endpoint}',
    URL_STYLE 'path',
    USE_SSL 'false'
);
""")

print("Connected to MinIO")

print("\nYahoo raw data (AAPL 1h):")
df_yahoo = conn.execute(
    "SELECT * FROM read_parquet('s3://raw-data/AAPL_1h.parquet') LIMIT 5"
).df()
display(df_yahoo)

print("\nBybit raw data (BTCUSDT 1h):")
df_bybit = conn.execute(
    "SELECT * FROM read_parquet('s3://raw-data/BTCUSDT_1h.parquet') LIMIT 5"
).df()
display(df_bybit)

conn_db = duckdb.connect("../data/analytics/financial_data.duckdb", read_only=True)

print("\nYahoo clean data by interval:")
yf_intervals = conn_db.execute(
    "SELECT DISTINCT interval FROM clean_yahoo_stocks ORDER BY interval"
).df()["interval"].tolist()

for interval in yf_intervals:
    print(f"\nInterval: {interval}")
    query = f"SELECT * FROM clean_yahoo_stocks WHERE interval = '{interval}' ORDER BY date LIMIT 5"
    display(conn_db.execute(query).df())

print("\nBybit clean data by interval:")
bybit_intervals = conn_db.execute(
    "SELECT DISTINCT interval FROM clean_bybit_crypto ORDER BY interval"
).df()["interval"].tolist()

for interval in bybit_intervals:
    print(f"\nInterval: {interval}")
    query = f"SELECT * FROM clean_bybit_crypto WHERE interval = '{interval}' ORDER BY date LIMIT 5"
    display(conn_db.execute(query).df())

conn_db.close()


Connected to MinIO

Yahoo raw data (AAPL 1h):


,date,close,high,low,open,volume
0,2024-04-03 13:30:00,170.036697,170.080002,168.589996,168.750000,8758184
1,2024-04-03 14:30:00,170.520004,170.679993,169.979996,170.024994,6176025
2,2024-04-03 15:30:00,170.320007,170.675003,170.199997,170.520004,3684320
3,2024-04-03 16:30:00,170.205002,170.440002,169.750000,170.309998,3812096
4,2024-04-03 17:30:00,170.197006,170.389999,170.120102,170.210007,3156608



Bybit raw data (BTCUSDT 1h):


,date,open,high,low,close,volume
0,2020-03-25 10:00:00,6500.0,6591.5,6500.0,6591.5,0.004
1,2020-03-25 11:00:00,6591.5,6628.5,6457.5,6511.5,438.873
2,2020-03-25 12:00:00,6511.5,6588.5,6502.0,6583.5,529.318
3,2020-03-25 13:00:00,6583.5,6745.5,6562.0,6585.0,449.162
4,2020-03-25 14:00:00,6585.0,6640.0,6516.0,6590.0,258.831



Yahoo clean data by interval:

Interval: 1d


...

,symbol,interval,date,open,high,low,close,volume
0,BTCUSDT,1h,2020-03-25 10:00:00,6500.0,6591.5,6500.0,6591.5,0.004
1,BTCUSDT,1h,2020-03-25 11:00:00,6591.5,6628.5,6457.5,6511.5,438.873
2,BTCUSDT,1h,2020-03-25 12:00:00,6511.5,6588.5,6502.0,6583.5,529.318
3,BTCUSDT,1h,2020-03-25 13:00:00,6583.5,6745.5,6562.0,6585.0,449.162
4,BTCUSDT,1h,2020-03-25 14:00:00,6585.0,6640.0,6516.0,6590.0,258.831



Interval: M


,symbol,interval,date,open,high,low,close,volume
0,BTCUSDT,M,2020-03-01,6500.0,6838.0,5841.5,6405.0,52238.985
1,BTCUSDT,M,2020-04-01,6405.0,9458.0,6145.0,8625.0,532029.546
2,BTCUSDT,M,2020-05-01,8625.0,10059.5,8153.0,9449.5,603928.977
3,BTCUSDT,M,2020-06-01,9449.5,10352.0,8840.0,9135.5,239068.501
4,BTCUSDT,M,2020-07-01,9135.5,11443.5,8910.0,11339.0,296758.679



Interval: W


,symbol,interval,date,open,high,low,close,volume
0,BTCUSDT,W,2020-03-23,6500.0,6838.0,5858.0,5873.0,21418.028
1,BTCUSDT,W,2020-03-30,5873.0,7230.5,5841.5,6771.0,114130.000
2,BTCUSDT,W,2020-04-06,6771.0,7458.5,6741.5,6901.0,111617.019
3,BTCUSDT,W,2020-04-13,6901.0,7289.0,6465.5,7118.5,114400.906
4,BTCUSDT,W,2020-04-20,7118.5,7743.5,6750.0,7691.5,110797.864
